# Lecture 6: the choice leaves the grid

**ECU44291 Quantitative Macroeconomics, Week 6.**

Two tools the grid was hiding: an **optimiser** (the choice $k'$ can be any number, not a grid point) and **interpolation** (the continuation value $V(k')$ has to be read between grid points). Both go inside value function iteration on the growth model, and the Euler-equation error from Lecture 5 measures what they buy.

This is the complete notebook, and I walk through it in the lecture. Before each code cell there is a question: answer it before you run the cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.optimize import minimize_scalar

## Task 1: golden-section search

Maximise $f(x) = 4x^2 e^{-2x}$ on $[0.1, 3]$ without derivatives. Write `golden_section(f, a, b, tol)`: two interior points at the golden ratio, keep the half that contains the higher one, reuse the surviving interior point. Return the maximiser and the list of brackets. Compare to `minimize_scalar(lambda x: -f(x), bounds=(0.1, 3), method="bounded")`.

**Question before running:** each step multiplies the width of the bracket by 0.618. Starting from a width of 2.9, about how many brackets until the width is below $10^{-8}$?

In [ ]:
# your code here


## Task 2: linear interpolation by hand

Write `interp_linear(x, xp, fp)`: find the bracket with `np.searchsorted`, weight the two endpoints, and clamp outside the grid. Check it against `np.interp` on and off the grid. Then evaluate it at `x = 5` when the grid ends at `1`: what comes back, and why is that dangerous inside a value function?

**Question before running:** what number comes back at `x = 5`?

In [ ]:
# your code here


## Task 3: why not a high-order polynomial?

Interpolate Runge's function $1/(1+25x^2)$ on 11 equispaced nodes in $[-1, 1]$ three ways: piecewise linear, `scipy.interpolate.CubicSpline`, and the degree-10 polynomial through the nodes (`scipy.interpolate.lagrange`). Plot all three against the truth. Which would you put inside a value function iteration, and why?

**Question before running:** which of the three will be furthest from the truth, and where?

In [ ]:
# your code here


## Task 4: fitted value function iteration

Lecture 4's growth model again ($\alpha = 0.3$, $\beta = 0.9$, log utility, full depreciation), but now $k'$ is continuous. In each sweep, build an interpolant of $V$ once, then for each grid point maximise $\ln(k^\alpha - k') + \beta\,\hat V(k')$ over $k' \in [k_{\min}, \min(k^\alpha, k_{\max})]$ with `minimize_scalar(method="bounded")`. Do it twice: with `np.interp` (linear) and with `scipy.interpolate.PchipInterpolator` (a smooth, shape-preserving cubic). For each, report the share of policy values that sit exactly on a grid point, and the maximum distance from the closed form $\alpha\beta k^\alpha$. Time both.

**Question before running:** the choice is now any number in a bracket. What share of the linear version's choices do you expect to land exactly on a grid point? And how long will each take, against a few milliseconds for the grid version?

In [ ]:
# your code here


Look at the share on nodes. With a linear interpolant the objective has a kink at every grid point, and a kink is where a concave objective's maximum tends to sit, so the optimiser "snaps" to nodes and the policy is still mostly a staircase. The smooth interpolant has no kinks and the policy lands on the closed form. **The interpolant is part of the method.**

**Workflow note.** *Measure before you optimise.* The fitted sweep is a hundred times slower. Before you touch it, `%time` the inner loop and find out where the time goes.

*NB:* `%time` at the start of a line runs that line and reports how long it took, e.g. `%time V, pol, n = fitted_vfi("linear")`. It works in notebooks only; in a `.py` file, use `time.perf_counter()` before and after, as in the cell above.

## Task 5: Euler-equation errors

The same measure as in Lecture 5. The Euler equation here is $1/c = \beta\alpha k'^{\alpha-1}/c'$. At points *between* grid points (midpoints), compute $c = k^\alpha - g(k)$, $k' = g(k)$, $c' = k'^\alpha - g(k')$ with $g$ interpolated, and the relative error $|1 - c\,\beta\alpha k'^{\alpha-1}/c'|$. Report $\log_{10}$ of the mean for three policies: the grid policy, fitted-linear, fitted-PCHIP. What does a value of $-4$ mean in words?

**Question before running:** in Lecture 5 the grid policy on 50 points scored $-2.10$. What will fitted-linear score? Fitted-PCHIP?

In [ ]:
# your code here


## The household, continuous: the M1 Part C task

The same two tools go inside the life-cycle household's age loop: `minimize_scalar` over $a' \in [0, \min(Ra + y_j, a_{\max})]$ in place of the argmax over the grid, with $V_{j+1}$ interpolated by PCHIP, built once per age. The slide shows what the result looks like. Building it is the M1 Part C task, due with the full submission on Monday 2 November at 09:00, so there is no code for it here.

## Before the next lecture

- **Monday 26 October, 23:59: the `m1-solo` tag.** The grid solver with your two features, your tests, the handwritten derivation and your notes. Nothing in this notebook is needed for it.
- **The study week: M1 Part C.** The continuous choice for your household, with an assistant and a log, and the critique. Keep the grid version: it is the check. Report the share of choices on nodes for both interpolants once, so you have seen the snap yourself, and add the Euler-residual check along the simulated path where the constraint does not bind. The full submission is due Monday 2 November at 09:00.
- Review method cards 4 (golden section) and 5 (linear interpolation).
- Workflow note: the continuous solver is much slower than the grid one. It's worth it to *Measure before you optimise.* to see this `%time` on the inner loop.
- Next lecture, Thursday 5 November: closing the model.